# Generate Methodology Mini-Plots

Creates compact figures directly from the locked radar, detector, ORDEM, calibration, and policy outputs for the methodology diagram.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


In [ ]:
from pathlib import Path
import hashlib
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROOT = PROJECT_ROOT
OUTPUT_DIR = ROOT / 'phase3_outputs' / 'methodology_mini_plots_v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCES = {
    'events': ROOT / 'phase0_outputs/catalog_matching/ca_cfar_1p75_24h_final_audited/final_consolidated_catalog_association.csv',
    'detector': ROOT / 'phase0_outputs/detector_benchmark_raw_swerling_v2/detector_operating_point_ranking.csv',
    'response': ROOT / 'phase1_outputs/ordem_action_library/b982_fine_elevation_49/ordem_fine_elevation_response_matrix_49.csv',
    'coverage': ROOT / 'phase3_outputs/publication_lock_validation/robust_lock_598bd992a783/coverage_reliability.csv',
    'learning': ROOT / 'phase3_outputs/publication_lock_validation/robust_lock_598bd992a783/learning_curve_summary.csv',
}

missing = [str(path) for path in SOURCES.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Required locked project outputs are missing:\n' + '\n'.join(missing))

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

for key, path in SOURCES.items():
    print(f'{key:>9}: {path.relative_to(ROOT)}  SHA256={sha256(path)[:12]}...')


In [ ]:
NAVY = '#071A4D'
BLUE = '#155EEF'
LIGHT_BLUE = '#77A7F7'
ORANGE = '#F05A18'
GRAY = '#6B7280'
LIGHT_GRAY = '#D9E1EC'
MINI_SIZE = (2.30, 1.85)
DPI = 450

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 6.5,
    'axes.labelsize': 7.0,
    'xtick.labelsize': 5.8,
    'ytick.labelsize': 5.8,
    'legend.fontsize': 5.4,
    'axes.linewidth': 0.75,
    'xtick.major.width': 0.65,
    'ytick.major.width': 0.65,
    'xtick.major.size': 2.5,
    'ytick.major.size': 2.5,
    'svg.fonttype': 'none',
})

def new_mini():
    fig, ax = plt.subplots(figsize=MINI_SIZE, facecolor='white')
    fig.subplots_adjust(left=0.22, bottom=0.24, right=0.96, top=0.95)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(direction='out')
    ax.grid(False)
    return fig, ax

def save_mini(fig, stem):
    png = OUTPUT_DIR / f'{stem}.png'
    svg = OUTPUT_DIR / f'{stem}.svg'
    fig.savefig(png, dpi=DPI, facecolor='white', transparent=False)
    fig.savefig(svg, facecolor='white', transparent=False)
    plt.close(fig)
    print(f'Saved {png.name} and {svg.name}')
    return png, svg


## 1. Actual 24-hour radar events


In [ ]:
events = pd.read_csv(SOURCES['events'])
required = {'reference_time_utc', 'observed_range_km', 'association_tier'}
if not required.issubset(events.columns):
    raise ValueError(f'Event file is missing columns: {sorted(required - set(events.columns))}')
events['time_utc'] = pd.to_datetime(events['reference_time_utc'], utc=True, errors='coerce')
events['range_km'] = pd.to_numeric(events['observed_range_km'], errors='coerce')
events = events.dropna(subset=['time_utc', 'range_km']).copy()
events['elapsed_h'] = (events['time_utc'] - events['time_utc'].min()).dt.total_seconds() / 3600.0
tier = events['association_tier'].fillna('unresolved').astype(str).str.lower()
events['catalog_associated'] = ~tier.str.contains('unresolved')

fig, ax = new_mini()
for flag, color, label, size, zorder in [
    (False, LIGHT_BLUE, 'Unresolved', 8, 1),
    (True, NAVY, 'Catalog-associated', 12, 2),
]:
    part = events[events['catalog_associated'] == flag]
    ax.scatter(part['elapsed_h'], part['range_km'], s=size, c=color, alpha=0.78,
               edgecolors='none', label=label, zorder=zorder)
ax.set_xlabel('Time from campaign start (h)')
ax.set_ylabel('One-way slant range (km)')
ax.set_xlim(-0.5, 24.5)
ax.set_xticks([0, 6, 12, 18, 24])
ax.legend(frameon=False, loc='best', handletextpad=0.25, borderpad=0.1)
save_mini(fig, '01_real_radar_events_range_time')
print(f'Plotted {len(events)} consolidated events; {events.catalog_associated.sum()} catalog-associated and {(~events.catalog_associated).sum()} unresolved.')


## 2. Actual CA-CFAR recovery heatmap


In [ ]:
detector = pd.read_csv(SOURCES['detector'])
required = {'method', 'threshold', 'track_rule', 'pd_swerling_all', 'false_tracks_per_24h_upper95'}
if not required.issubset(detector.columns):
    raise ValueError(f'Detector file is missing columns: {sorted(required - set(detector.columns))}')
short = detector[detector['track_rule'].astype(str).str.lower().eq('short_0p8')].copy()
if short.empty:
    raise ValueError('No short_0p8 detector candidates found; refusing to substitute another track rule.')
short['method_key'] = (short['method'].astype(str).str.lower()
                       .str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_'))
short['threshold_num'] = pd.to_numeric(short['threshold'], errors='coerce')
model_columns = ['pd_nonfluctuating', 'pd_swerling1', 'pd_swerling2', 'pd_swerling3', 'pd_swerling4']
missing_models = [c for c in model_columns if c not in short.columns]
if missing_models:
    raise ValueError(f'Detector file is missing recovery columns: {missing_models}')
ca = short[short['method_key'].eq('ca_cfar')].dropna(subset=['threshold_num']).copy()
if ca.empty:
    raise ValueError('No CA-CFAR short-track benchmark rows found.')
ca[model_columns] = ca[model_columns].apply(pd.to_numeric, errors='coerce')
ca = ca.dropna(subset=model_columns)
ca = ca.groupby('threshold_num', as_index=False)[model_columns].mean().sort_values('threshold_num')
thresholds = ca['threshold_num'].to_numpy()
recovery = ca[model_columns].to_numpy(dtype=float).T
if np.nanmax(recovery) > 1.5:
    recovery = recovery / 100.0
if len(thresholds) < 2:
    raise ValueError('At least two tested CA-CFAR thresholds are required for a heatmap.')

fig, ax = new_mini()
image = ax.imshow(
    recovery, origin='lower', aspect='auto', interpolation='bilinear', cmap='magma',
    vmin=0.0, vmax=max(0.01, float(np.nanmax(recovery))),
    extent=[thresholds.min(), thresholds.max(), -0.5, len(model_columns) - 0.5]
)
ax.axvline(1.75, color=ORANGE, lw=1.25, ls='--')
ax.set_xlabel('CA-CFAR threshold')
ax.set_ylabel('Injected target model')
ax.set_yticks(range(len(model_columns)))
ax.set_yticklabels(['Steady', 'S-I', 'S-II', 'S-III', 'S-IV'])
ax.set_xticks([thresholds.min(), 1.75, thresholds.max()])
cbar = fig.colorbar(image, ax=ax, fraction=0.055, pad=0.03)
cbar.ax.tick_params(labelsize=5.2, length=2)
cbar.set_label('Recovery probability', fontsize=6.1)
selected = ca[np.isclose(ca['threshold_num'], 1.75, equal_nan=False)]
if selected.empty:
    raise ValueError('Selected CA-CFAR threshold 1.75 is absent; refusing to guess a replacement.')
save_mini(fig, '02_real_cfar_swerling_recovery_heatmap')
print(f'Plotted {len(thresholds)} tested CA-CFAR thresholds across five injected target models.')


## 3. Actual elevation-altitude response library


In [ ]:
response = pd.read_csv(SOURCES['response'])
shell_cols = [c for c in response.columns if c.startswith('shell_') and c.endswith('_km')]
if 'elevation_deg' not in response.columns or not shell_cols:
    raise ValueError('Response file lacks elevation_deg or shell_*_km columns.')
shell_altitudes = np.array([float(re.search(r'shell_([0-9.]+)_km', c).group(1)) for c in shell_cols])
elevations = pd.to_numeric(response['elevation_deg'], errors='coerce').to_numpy()
matrix = response[shell_cols].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
if np.isnan(elevations).any() or np.isnan(matrix).any():
    raise ValueError('Non-numeric values found in the locked response matrix.')
row_sums = matrix.sum(axis=1, keepdims=True)
if np.any(row_sums <= 0):
    raise ValueError('At least one response row has zero or negative total response.')
relative = matrix / row_sums

fig, ax = new_mini()
image = ax.imshow(
    relative.T, origin='lower', aspect='auto', interpolation='nearest', cmap='viridis',
    extent=[elevations.min(), elevations.max(), shell_altitudes.min(), shell_altitudes.max()]
)
ax.set_xlabel('Radar elevation (degrees)')
ax.set_ylabel('Altitude shell (km)')
ax.set_xticks([65, 75, 85])
ax.set_yticks([922.5, 982.5, 1057.5])
cbar = fig.colorbar(image, ax=ax, fraction=0.055, pad=0.03)
cbar.ax.tick_params(labelsize=5.2, length=2)
cbar.set_label('Relative response', fontsize=6.1)
save_mini(fig, '03_real_elevation_altitude_response')
print(f'Plotted the locked {len(elevations)} elevation x {len(shell_altitudes)} altitude-shell response library.')


## 4. Actual Bayesian uncertainty calibration


In [ ]:
coverage = pd.read_csv(SOURCES['coverage'])
required = {
    'experiment_block', 'scenario', 'strength', 'response_correlation_deg', 'policy',
    'nominal_level', 'observed_coverage', 'ci95_low', 'ci95_high'
}
if not required.issubset(coverage.columns):
    raise ValueError(f'Coverage file is missing columns: {sorted(required - set(coverage.columns))}')
cal = coverage[
    coverage['experiment_block'].astype(str).eq('main')
    & coverage['scenario'].astype(str).eq('reported_response_uncertainty')
    & np.isclose(pd.to_numeric(coverage['strength'], errors='coerce'), 1.0)
    & np.isclose(pd.to_numeric(coverage['response_correlation_deg'], errors='coerce'), 1.5)
    & coverage['policy'].astype(str).eq('asset_aopt')
].copy()
if cal.empty:
    raise ValueError('Locked primary adaptive-policy coverage rows were not found.')
cal = cal.sort_values('nominal_level')
x = pd.to_numeric(cal['nominal_level'], errors='coerce').to_numpy()
y = pd.to_numeric(cal['observed_coverage'], errors='coerce').to_numpy()
lo = pd.to_numeric(cal['ci95_low'], errors='coerce').to_numpy()
hi = pd.to_numeric(cal['ci95_high'], errors='coerce').to_numpy()
if np.isnan(np.r_[x, y, lo, hi]).any():
    raise ValueError('Non-numeric values found in the locked coverage rows.')

fig, ax = new_mini()
axis_min = max(0.0, min(x.min(), lo.min()) - 0.04)
ax.plot([axis_min, 1.0], [axis_min, 1.0], color=GRAY, lw=1.0, ls='--', label='Ideal')
ax.fill_between(x, lo, hi, color=LIGHT_BLUE, alpha=0.28, linewidth=0)
ax.plot(x, y, color=BLUE, lw=1.8, marker='o', ms=3.0, label='Adaptive')
ax.set_xlabel('Nominal interval probability')
ax.set_ylabel('Observed truth coverage')
ax.set_xlim(axis_min, 1.0)
ax.set_ylim(axis_min, 1.0)
ax.set_aspect('equal', adjustable='box')
ax.legend(frameon=False, loc='upper left', handletextpad=0.3, borderpad=0.1)
save_mini(fig, '04_real_bayesian_coverage_calibration')
print(f'Plotted {len(cal)} nominal coverage levels for the locked primary adaptive-policy configuration.')


## 5. Actual adaptive-versus-baseline learning curves


In [ ]:
learning = pd.read_csv(SOURCES['learning'])
required = {
    'experiment_block', 'scenario', 'strength', 'response_correlation_deg', 'policy', 'step',
    'asset_uncertainty_fraction_mean', 'asset_uncertainty_fraction_ci95_low',
    'asset_uncertainty_fraction_ci95_high'
}
if not required.issubset(learning.columns):
    raise ValueError(f'Learning file is missing columns: {sorted(required - set(learning.columns))}')
primary = learning[
    learning['experiment_block'].astype(str).eq('main')
    & learning['scenario'].astype(str).eq('reported_response_uncertainty')
    & np.isclose(pd.to_numeric(learning['strength'], errors='coerce'), 1.0)
    & np.isclose(pd.to_numeric(learning['response_correlation_deg'], errors='coerce'), 1.5)
].copy()

policy_order = ['asset_aopt', 'best_fixed', 'random']
policy_labels = {'asset_aopt': 'Adaptive', 'best_fixed': 'Best fixed', 'random': 'Random'}
policy_colors = {'asset_aopt': ORANGE, 'best_fixed': NAVY, 'random': GRAY}
policy_styles = {'asset_aopt': '-', 'best_fixed': '--', 'random': ':'}
missing_policies = [p for p in policy_order if p not in set(primary['policy'].astype(str))]
if missing_policies:
    raise ValueError(f'Locked primary learning curves are missing policies: {missing_policies}')

fig, ax = new_mini()
for policy in policy_order:
    part = primary[primary['policy'].astype(str).eq(policy)].sort_values('step')
    step = pd.to_numeric(part['step'], errors='coerce').to_numpy()
    mean = pd.to_numeric(part['asset_uncertainty_fraction_mean'], errors='coerce').to_numpy()
    lo = pd.to_numeric(part['asset_uncertainty_fraction_ci95_low'], errors='coerce').to_numpy()
    hi = pd.to_numeric(part['asset_uncertainty_fraction_ci95_high'], errors='coerce').to_numpy()
    if np.isnan(np.r_[step, mean, lo, hi]).any():
        raise ValueError(f'Non-numeric values found in learning curve for {policy}.')
    ax.fill_between(step, lo, hi, color=policy_colors[policy], alpha=0.10, linewidth=0)
    ax.plot(step, mean, color=policy_colors[policy], ls=policy_styles[policy],
            lw=1.65, label=policy_labels[policy])
ax.set_xlabel('Observation number')
ax.set_ylabel('Remaining weighted uncertainty')
ax.set_xlim(0, 24)
ax.set_xticks([0, 8, 16, 24])
ax.set_ylim(bottom=0)
ax.legend(frameon=False, loc='best', handlelength=2.1, handletextpad=0.35, borderpad=0.1)
save_mini(fig, '05_real_policy_learning_curves')
print('Plotted the locked primary adaptive, best-fixed, and random learning curves.')


## Contact sheet, source manifest, and completion marker


In [ ]:
plot_specs = [
    ('01_real_radar_events_range_time', 'A  Real radar events', 'events',
     'Measured event time and one-way slant range; color is association status.'),
    ('02_real_cfar_swerling_recovery_heatmap', 'B  Detector validation', 'detector',
     'CA-CFAR injected-target recovery across tested thresholds and five target-fluctuation models.'),
    ('03_real_elevation_altitude_response', 'C  Pointing response', 'response',
     'Row-normalized ORDEM response across 49 elevations and 10 altitude shells.'),
    ('04_real_bayesian_coverage_calibration', 'D  Uncertainty calibration', 'coverage',
     'Nominal versus observed known-truth interval coverage for the locked primary adaptive setup.'),
    ('05_real_policy_learning_curves', 'E  Policy comparison', 'learning',
     'Remaining weighted posterior uncertainty versus observation number.'),
]

fig, axes = plt.subplots(2, 3, figsize=(10.8, 6.0), facecolor='white')
axes = axes.ravel()
for ax, (stem, title, _, _) in zip(axes, plot_specs):
    image = plt.imread(OUTPUT_DIR / f'{stem}.png')
    ax.imshow(image)
    ax.set_title(title, fontsize=10, color=NAVY, fontweight='bold', pad=6)
    ax.axis('off')
axes[-1].axis('off')
fig.tight_layout(pad=1.1)
contact_sheet = OUTPUT_DIR / '00_methodology_real_mini_plots_contact_sheet.png'
fig.savefig(contact_sheet, dpi=250, facecolor='white', bbox_inches='tight')
plt.close(fig)

manifest_rows = []
for index, (stem, title, source_key, meaning) in enumerate(plot_specs, start=1):
    source = SOURCES[source_key]
    manifest_rows.append({
        'plot_number': index,
        'plot_title': title,
        'png_file': f'{stem}.png',
        'svg_file': f'{stem}.svg',
        'source_relative_to_research': str(source.relative_to(ROOT)),
        'source_sha256': sha256(source),
        'meaning': meaning,
    })
manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(OUTPUT_DIR / 'methodology_mini_plot_manifest.csv', index=False)

completion = {
    'status': 'SUCCESS',
    'output_directory': str(OUTPUT_DIR),
    'plot_count': len(plot_specs),
    'png_and_svg_exported': True,
    'synthetic_or_placeholder_data_used': False,
    'source_files': {key: str(path.relative_to(ROOT)) for key, path in SOURCES.items()},
}
with open(OUTPUT_DIR / '_SUCCESS.json', 'w') as handle:
    json.dump(completion, handle, indent=2)

print(f'Contact sheet: {contact_sheet}')
print(f'Manifest: {OUTPUT_DIR / "methodology_mini_plot_manifest.csv"}')
print('SUCCESS - five real methodology mini-plots generated; no placeholder data used.')
